<a href="https://colab.research.google.com/github/vinipi/ailead_gpumanagement/blob/main/exercise_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import torch
import torch.nn as nn

platform = __import__("platform")  # platform is a standard library module

print("Python platform:", platform.platform())
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device count:", torch.cuda.device_count())
    print("Current CUDA device:", torch.cuda.current_device())
    print("CUDA device name:", torch.cuda.get_device_name(0))
    # Get the total VRAM in GB
    props = torch.cuda.get_device_properties(0)
    total_vram_gb = props.total_memory / 1e9
    print(f"Total VRAM: {total_vram_gb:.1f} GB")

# Check Apple MPS in two clear steps
has_mps_attribute = hasattr(torch.backends, "mps")
if has_mps_attribute:
    mps_available = torch.backends.mps.is_available()
else:
    mps_available = False

print("MPS available:", mps_available)




Python platform: Linux-6.6.122+-x86_64-with-glibc2.35
PyTorch version: 2.11.0+cu128
CUDA available: True
CUDA device count: 1
Current CUDA device: 0
CUDA device name: Tesla T4
Total VRAM: 15.6 GB
MPS available: False


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if mps_available else "cpu")

print("Using device:", device)

# tensor of shape (3, 224, 224) on the CPU
x = torch.randn(3, 224, 224)

print("Before:", x.device)

x = x.to(device)

print("After:", x.device)


Using device: cuda
Before: cpu
After: cuda:0


In [11]:

# Linear model with 784 input features, 128 hidden units, and 10 output classes

simple_model = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)

# Look at the device of the first weight to know where the model lives
model_parameters = simple_model.parameters()
first_parameter = next(model_parameters)
print("Before:", first_parameter.device)

simple_model = simple_model.to(device)

model_parameters = simple_model.parameters()
first_parameter = next(model_parameters)
print("After:", first_parameter.device)


Before: cpu
After: cuda:0


In [15]:
#try the model
# Create one fake 28x28 image = 784 values
x = torch.randn(1, 784).to(device)

# Give the tensor to the model
output = simple_model(x)

print("Input shape:", x.shape)
print("Output:", output)
print("Output shape:", output.shape)



Input shape: torch.Size([1, 784])
Output: tensor([[-0.0274, -0.2800, -0.3558,  0.1922,  0.2166,  0.0247, -0.0310,  0.1255,
          0.2264,  0.2335]], device='cuda:0', grad_fn=<AddmmBackward0>)
Output shape: torch.Size([1, 10])


In [9]:
# This cell intentionally creates a common error on CUDA.
# Run it only if you want to see the error.
# Then fix it by moving x to the same device as the model.

run_broken_example = False  # Change to True to see the error

if run_broken_example and torch.cuda.is_available():
    broken_model = nn.Linear(10, 2)
    broken_model = broken_model.to("cuda")

    x = torch.randn(4, 10)  # CPU tensor

    # This should fail because the model is on CUDA and x is on CPU.
    output = broken_model(x)


In [16]:
if torch.cuda.is_available():
    device_count = torch.cuda.device_count()
    print("Number of CUDA devices:", device_count)

    for i in range(device_count):
        device_name = torch.cuda.get_device_name(i)
        print(f"Device {i}: {device_name}")
else:
    print("CUDA is not available.")


Number of CUDA devices: 1
Device 0: Tesla T4


In [ ]:
!nvidia-smi -l 1

In [20]:
if torch.cuda.is_available():
    allocated_bytes = torch.cuda.memory_allocated()
    allocated_gb = allocated_bytes / 1e9
    print("Allocated memory:", allocated_gb, "GB")

    reserved_bytes = torch.cuda.memory_reserved()
    reserved_gb = reserved_bytes / 1e9
    print("Reserved memory:", reserved_gb, "GB")
else:
    print("CUDA is not available. Memory inspection skipped.")


Allocated memory: 0.010388992 GB
Reserved memory: 0.023068672 GB


In [21]:
if torch.cuda.is_available():
    print("Before tensor creation")
    allocated_bytes = torch.cuda.memory_allocated()
    allocated_gb = allocated_bytes / 1e9
    reserved_bytes = torch.cuda.memory_reserved()
    reserved_gb = reserved_bytes / 1e9
    print("Allocated:", allocated_gb, "GB")
    print("Reserved:", reserved_gb, "GB")

    # Make a large tensor of 100 million elements on the GPU on purpose, so we use real VRAM
    big_tensor = torch.randn(10_000, 10_000, device="cuda")

    print("\nAfter tensor creation")
    allocated_bytes = torch.cuda.memory_allocated()
    allocated_gb = allocated_bytes / 1e9
    reserved_bytes = torch.cuda.memory_reserved()
    reserved_gb = reserved_bytes / 1e9
    print("Allocated:", allocated_gb, "GB")
    print("Reserved:", reserved_gb, "GB")

    # Delete the tensor and release the cached memory back to the GPU
    del big_tensor
    torch.cuda.empty_cache()

    print("\nAfter deleting tensor and emptying cache")
    allocated_bytes = torch.cuda.memory_allocated()
    allocated_gb = allocated_bytes / 1e9
    reserved_bytes = torch.cuda.memory_reserved()
    reserved_gb = reserved_bytes / 1e9
    print("Allocated:", allocated_gb, "GB")
    print("Reserved:", reserved_gb, "GB")
else:
    print("CUDA is not available. Run this cell on a GPU runtime.")


Before tensor creation
Allocated: 0.010388992 GB
Reserved: 0.023068672 GB

After tensor creation
Allocated: 0.410945024 GB
Reserved: 0.423624704 GB

After deleting tensor and emptying cache
Allocated: 0.010388992 GB
Reserved: 0.023068672 GB


In [23]:
import gc

# Force Python to clean up
gc.collect()

# Release PyTorch's reserved GPU cache
torch.cuda.empty_cache()